# CT3 Data QC, Sensitivity and Feature Ablation

## Goal

This notebook validates the 199 exported FEM cases, records provisional
physical-boundary flags, measures development-case sensitivity, and runs
a complete-case feature ablation before symbolic regression.

A **case** is the independent operating condition.  Every element in a
case stays together.  No element-level train/test split and no element
sampling are used.

The exported coordinates are treated as post-deformation predictor
inputs.  Consequently, the resulting surrogate is an independent stress
predictor only for future situations where the same post-deformation
coordinates and the three parameter fields are available before stress is
requested.

The 50-case final test is structurally inventoried but excluded from
sensitivity interpretation and feature-set selection.


## 1. Setup and frozen case-level split


In [ ]:
from pathlib import Path
import gc
import json
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
PACKAGE_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'src' / 'ct3_common.py').is_file()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory')
sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from ct3_common import *

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
warnings.filterwarnings("ignore")

CASE_DIR_OVERRIDE = None
PATHS = build_paths(PACKAGE_ROOT, CASE_DIR_OVERRIDE)
CASE_FILES, CASE_INVENTORY = discover_case_files(PATHS.case_dir)
CASE_PATH_BY_ID = case_path_lookup(CASE_INVENTORY)
SPLIT_MANIFEST = load_frozen_manifest(PATHS.manifest_path, CASE_INVENTORY)
DEVELOPMENT_CASE_IDS = development_case_ids(SPLIT_MANIFEST)
FINAL_CASE_IDS = final_test_case_ids(SPLIT_MANIFEST)

print("Package root:", PATHS.package_root)
print("Case directory:", PATHS.case_dir)
print("Cases:", len(CASE_INVENTORY))
print("Development cases:", len(DEVELOPMENT_CASE_IDS))
print("Locked final-test cases:", len(FINAL_CASE_IDS))
print("Random seed:", RANDOM_SEED)

OUTPUT_DIR = PATHS.output_root / "00_qc_sensitivity_ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_DEVELOPMENT_SENSITIVITY = True
RUN_FULL_ABLATION = True
ABLATION_MODEL_MAX_ITER = 180

CASE_INVENTORY.to_csv(OUTPUT_DIR / "case_file_inventory.csv", index=False)
SPLIT_MANIFEST.to_csv(OUTPUT_DIR / "frozen_manifest_copy.csv", index=False)
display(SPLIT_MANIFEST.groupby(["iteration", "split"]).size().unstack(fill_value=0))


## 2. Full development-case sensitivity

Spearman correlation is computed separately inside every development
case.  It measures monotonic association, not causality.  Aggregating the
absolute correlation and its sign stability across 149 cases helps expose
relationships that are persistent rather than driven by one FEM case.

Boundary checks only flag values.  They do not clip, replace, or delete
observations.  The current provisional ranges are FluenceRate 0-7,
Temperature 25-600 C, WeightLossRate 0-2 %/yr, and stress approximately
-70 to +35 pending the professor's stress-sign decision.


In [ ]:
sensitivity_features = list(dict.fromkeys(
    PHYSICAL_FEATURES + RAW_POSITION_FEATURES + ["theta_sin", "theta_cos"]
))
summary_records = []
correlation_records = []
boundary_records = []
read_audit_records = []

if RUN_DEVELOPMENT_SENSITIVITY:
    for index, case_id in enumerate(DEVELOPMENT_CASE_IDS, start=1):
        print(f"[{index}/{len(DEVELOPMENT_CASE_IDS)}] Sensitivity: {case_id}")
        frame, audit = read_complete_case(CASE_PATH_BY_ID[case_id])
        read_audit_records.append(audit)
        boundary_records.extend(physical_boundary_audit(frame, case_id))

        target = frame[TARGET_COL]
        record = {
            "case_id": case_id,
            "case_number": audit["case_number"],
            "n_elements": len(frame),
            "stress_mean": float(target.mean()),
            "stress_std": float(target.std(ddof=0)),
            "stress_p95": float(target.quantile(0.95)),
            "stress_p99": float(target.quantile(0.99)),
            "stress_max": float(target.max()),
            "negative_stress_fraction": float((target < 0).mean()),
        }
        for feature in sensitivity_features:
            values = frame[feature]
            record.update({
                f"{feature}_mean": float(values.mean()),
                f"{feature}_std": float(values.std(ddof=0)),
                f"{feature}_min": float(values.min()),
                f"{feature}_p95": float(values.quantile(0.95)),
                f"{feature}_max": float(values.max()),
            })
            correlation_records.append({
                "case_id": case_id,
                "case_number": audit["case_number"],
                "feature": feature,
                "spearman": float(frame[feature].corr(target, method="spearman")),
                "pearson": float(frame[feature].corr(target, method="pearson")),
                "n_elements": len(frame),
            })
        summary_records.append(record)
        del frame
        gc.collect()

    case_summary = pd.DataFrame(summary_records).sort_values("case_number")
    correlations = pd.DataFrame(correlation_records)
    boundary_audit = pd.DataFrame(boundary_records)
    read_audit = pd.DataFrame(read_audit_records).sort_values("case_number")
    sensitivity_stability = (
        correlations.assign(
            abs_spearman=lambda x: x["spearman"].abs(),
            positive_spearman=lambda x: x["spearman"] > 0,
        )
        .groupby("feature", observed=True)
        .agg(
            n_cases=("case_id", "nunique"),
            mean_spearman=("spearman", "mean"),
            median_spearman=("spearman", "median"),
            mean_abs_spearman=("abs_spearman", "mean"),
            std_spearman=("spearman", "std"),
            positive_sign_fraction=("positive_spearman", "mean"),
        )
        .reset_index()
        .sort_values("mean_abs_spearman", ascending=False)
    )

    case_summary.to_csv(OUTPUT_DIR / "development_case_summary.csv", index=False)
    correlations.to_csv(OUTPUT_DIR / "within_case_feature_correlations.csv", index=False)
    sensitivity_stability.to_csv(OUTPUT_DIR / "cross_case_sensitivity_stability.csv", index=False)
    boundary_audit.to_csv(OUTPUT_DIR / "provisional_boundary_audit.csv", index=False)
    read_audit.to_csv(OUTPUT_DIR / "complete_case_read_audit.csv", index=False)
    display(sensitivity_stability)
    display(boundary_audit.groupby("variable").agg(
        cases_flagged=("fraction_outside", lambda x: int((x > 0).sum())),
        maximum_fraction_outside=("fraction_outside", "max"),
    ))
else:
    print("Development sensitivity was skipped.")


## 3. Complete-case feature ablation

HistGradientBoosting is used here as a nonlinear diagnostic, not as the
final model.  Four input views are compared under the same four frozen
case splits and the same upper-tail weights:

- physical fields only;
- position only, used only as a confounding diagnostic;
- physical fields plus raw rho/theta/z;
- physical fields plus rho/sin(theta)/cos(theta)/z.

All 119 training cases and every element are used in each round.  Feature
selection uses validation cases only.  Internal-test metrics are reported
after selection and the 50 final cases are never loaded here.  The final
set is locked from CT3 modelling, although its stress summaries were
inspected during earlier QC and boundary-anchor split construction.


In [ ]:
try:
    from sklearn.ensemble import HistGradientBoostingRegressor
except Exception as exc:
    raise ImportError("scikit-learn is required for the ablation stage") from exc

ALL_ABLATION_FEATURES = list(dict.fromkeys(
    FEATURE_SETS["full_raw_coordinates"] + FEATURE_SETS["full_periodic_coordinates"]
))

def evaluate_model_cases(model, case_ids, feature_indices, iteration, feature_set, split):
    rows = []
    for case_id in case_ids:
        payload, _ = load_case_payload(case_id, CASE_PATH_BY_ID, ALL_ABLATION_FEATURES)
        predicted = model.predict(payload["X"][:, feature_indices])
        rows.append({
            "iteration": iteration,
            "feature_set": feature_set,
            "split": split,
            "case_id": case_id,
            **evaluate_prediction_arrays(payload["y"], predicted),
        })
        del payload, predicted
        gc.collect()
    return rows

ablation_case_records = []
ablation_training_audit_records = []

if RUN_FULL_ABLATION:
    for iteration in range(1, 5):
        train_ids = cases_for_role(SPLIT_MANIFEST, iteration, "train")
        validation_ids = cases_for_role(SPLIT_MANIFEST, iteration, "validation")
        internal_ids = cases_for_role(SPLIT_MANIFEST, iteration, "internal_test")
        assert_no_final_cases(train_ids + validation_ids + internal_ids, SPLIT_MANIFEST)

        print(f"Iteration {iteration}: assembling all {len(train_ids)} training cases")
        bundle = assemble_full_training_arrays(
            train_ids, CASE_PATH_BY_ID, ALL_ABLATION_FEATURES,
            scale=False, use_tail_weights=True,
        )
        bundle["audit"].assign(iteration=iteration).to_csv(
            OUTPUT_DIR / f"ablation_training_audit_iteration_{iteration}.csv", index=False
        )

        for feature_set, features in FEATURE_SETS.items():
            indices = [ALL_ABLATION_FEATURES.index(feature) for feature in features]
            model = HistGradientBoostingRegressor(
                loss="squared_error",
                learning_rate=0.05,
                max_iter=ABLATION_MODEL_MAX_ITER,
                max_leaf_nodes=63,
                min_samples_leaf=256,
                l2_regularization=1.0,
                early_stopping=False,
                random_state=RANDOM_SEED,
            )
            print(f"  Fitting {feature_set}: {features}")
            model.fit(
                bundle["X"][:, indices], bundle["y"],
                sample_weight=bundle["weights"],
            )
            ablation_case_records.extend(evaluate_model_cases(
                model, validation_ids, indices, iteration, feature_set, "validation"
            ))
            ablation_case_records.extend(evaluate_model_cases(
                model, internal_ids, indices, iteration, feature_set, "internal_test"
            ))
            pd.DataFrame(ablation_case_records).to_csv(
                OUTPUT_DIR / "ablation_case_metrics.csv", index=False
            )
            del model
            gc.collect()
        del bundle
        gc.collect()

    ablation_case_metrics = pd.DataFrame(ablation_case_records)
    ablation_split_metrics = aggregate_case_metrics(
        ablation_case_metrics, ["iteration", "feature_set", "split"]
    )
    metric_columns = [metric for metric, _, _ in SELECTION_METRICS]
    ablation_stability = (
        ablation_split_metrics.groupby(["feature_set", "split"])[metric_columns]
        .agg(["mean", "std", "min", "max"])
    )
    ablation_stability.columns = [f"{metric}_{stat}" for metric, stat in ablation_stability.columns]
    ablation_stability = ablation_stability.reset_index()

    validation_average = (
        ablation_split_metrics[ablation_split_metrics["split"] == "validation"]
        .groupby("feature_set", observed=True)[metric_columns]
        .mean()
        .reset_index()
    )
    validation_ranked = add_engineering_selection_score(validation_average)
    validation_ranked["eligible_for_symbolic_regression"] = (
        validation_ranked["feature_set"] != "position_only_periodic"
    )
    eligible = validation_ranked[validation_ranked["eligible_for_symbolic_regression"]].copy()
    selected = eligible.sort_values([
        "engineering_selection_score", "macro_rmse", "mean_p99_relative_error", "feature_set"
    ]).iloc[0]
    selected_features = FEATURE_SETS[selected["feature_set"]]
    locked_selection = pd.DataFrame([{
        "feature_set": selected["feature_set"],
        "features_json": json.dumps(selected_features),
        "selection_source": "four_round_validation_only_histgradientboosting_ablation",
        "engineering_selection_score": selected["engineering_selection_score"],
        "random_seed": RANDOM_SEED,
        "final_test_used_for_selection": False,
        "position_only_is_diagnostic_not_eligible": True,
    }])

    ablation_split_metrics.to_csv(OUTPUT_DIR / "ablation_split_metrics.csv", index=False)
    ablation_stability.to_csv(OUTPUT_DIR / "ablation_stability_summary.csv", index=False)
    validation_ranked.to_csv(OUTPUT_DIR / "ablation_validation_ranking.csv", index=False)
    locked_selection.to_csv(OUTPUT_DIR / "locked_feature_set.csv", index=False)
    display(validation_ranked.sort_values("engineering_selection_score"))
    display(locked_selection)
else:
    print("Full ablation was skipped. No feature set was locked.")


## Takeaways

`locked_feature_set.csv` is the only feature-choice interface used by
later notebooks.  Position-only performance is interpreted as evidence
of spatial confounding, not as permission to remove the physical fields.
The feature choice remains a predictive decision and must not be described
as causal sensitivity.
